In [2]:
!nvidia-smi

Mon Apr 20 19:36:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [4]:
import os, getpass
token = getpass.getpass("GitHub token: ")
!git clone https://{token}@github.com/jasmineztruong8/efficient-codegen.git
%cd /content/efficient-codegen

!git fetch origin
!git checkout jg/serving-benchmarks
!git pull origin jg/serving-benchmarks
!git log --oneline -n 10


GitHub token: ··········
Cloning into 'efficient-codegen'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 226 (delta 35), reused 46 (delta 13), pack-reused 140 (from 1)
Receiving objects: 100% (226/226), 25.87 MiB | 11.47 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/efficient-codegen
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 3), reused 10 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 18.13 KiB | 3.63 MiB/s, done.
From https://github.com/jasmineztruong8/efficient-codegen
 * [new branch]      jg/serving-benchmarks -> origin/jg/serving-benchmarks
Branch 'jg/serving-benchmarks' set up to track remote branch 'jg/serving-benchmarks' from 'origin'.
Switched to a new branch 'jg/serving-benchmarks'
From https://github.com/jasmineztruong8/efficie

In [5]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl torch pandas
!pip install -q vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 152.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
RUNTIME_AWARE_ADAPTER = "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full"
RUNTIME_AWARE_MERGED = "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged"

INPUT_PATH = "data/curated/test/dataset_clean.json"
SERVING_OUTPUT_DIR = "outputs/serving"

WANDB_PROJECT = "hpml-efficient-codegen"

In [9]:
!ls serving
!ls training
!ls data/curated/test

benchmark_serving.py  merge_checkpoint.py
data  evaluate_model.py  select_training_data.py  train.py
benchmark_results.json	dataset_clean.json


In [10]:
!python serving/merge_checkpoint.py \
  --adapter_path {RUNTIME_AWARE_ADAPTER} \
  --base_model_name {BASE_MODEL} \
  --output_dir {RUNTIME_AWARE_MERGED}

Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 7.30kB [00:00, 12.9MB/s]
vocab.json: 2.78MB [00:00, 112MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 7.03MB [00:00, 37.8MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:08<00:00, 366MB/s]
Loading weights: 100% 338/338 [00:00<00:00, 421.01it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.28MB/s]
Loading adapter: /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full
Merging adapter weights into base model...
Saving merged model to /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
Writing model shards: 100% 1/1 [00:09<00:00,  9.84s/it]
Done.


In [11]:
# smoke: base + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/base_hf_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_hf_smoke


Loaded 100 prompts
Running Hugging Face benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 438.51it/s]
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.8423035143199991,
  "avg_batch_latency_s": 6.475257196153839,
  "throughput_prompts_per_s": 1.1872205006853271,
  "throughput_output_tokens_per_s": 217.02390752527782,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.79503105590062,
  "max_gpu_util_pct": 55.0,
  "avg_gpu_mem_mb": 4134.6459627329195,
  "max_gpu_mem_mb": 4150.0
}
Results saved to outputs/serving/base_hf_smoke.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --r

In [12]:
# smoke: base + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/base_vllm_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_vllm_smoke


Loaded 100 prompts
Running vLLM benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
INFO 04-20 19:49:33 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}
INFO 04-20 19:49:50 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 19:49:50 [model.py:1678] Using max model len 32768
INFO 04-20 19:49:50 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 19:49:50 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 19:49:52 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=10228) INFO 04-20 19:50:06 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen2.5-Coder-1.5B-Instruct', sp

In [13]:
# smoke: runtime aware + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_hf_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_hf_smoke

Loaded 100 prompts
Running Hugging Face benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 175.34it/s]
{
  "backend": "hf",
  "model_path": "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.7775845633000017,
  "avg_batch_latency_s": 5.977211773307632,
  "throughput_prompts_per_s": 1.2860337604389758,
  "throughput_output_tokens_per_s": 220.16897978715267,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.95302013422819,
  "max_gpu_util_pct": 45.0,
  "avg_gpu_mem_mb": 4139.463087248322,
  "max_gpu_mem_mb": 4150.0
}
Results saved to outputs/serving/runtime_aware_hf_smoke.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logge

In [14]:
# smoke: runtime aware + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 100 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_vllm_smoke.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_vllm_smoke

Loaded 100 prompts
Running vLLM benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
INFO 04-20 19:53:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged'}
INFO 04-20 19:53:10 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 19:53:10 [model.py:1678] Using max model len 32768
INFO 04-20 19:53:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 19:53:10 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 19:53:13 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=11822) INFO 04-20 19:53:26 [core.py:105] Initializing a

In [15]:
!ls {SERVING_OUTPUT_DIR}
!cat {SERVING_OUTPUT_DIR}/base_hf_smoke.json
!cat {SERVING_OUTPUT_DIR}/base_vllm_smoke.json
!cat {SERVING_OUTPUT_DIR}/runtime_aware_hf_smoke.json
!cat {SERVING_OUTPUT_DIR}/runtime_aware_vllm_smoke.json

base_hf_smoke.json    runtime_aware_hf_smoke.json
base_vllm_smoke.json  runtime_aware_vllm_smoke.json
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.8423035143199991,
  "avg_batch_latency_s": 6.475257196153839,
  "throughput_prompts_per_s": 1.1872205006853271,
  "throughput_output_tokens_per_s": 217.02390752527782,
  "peak_cuda_memory_mb": 3507.087890625,
  "avg_gpu_util_pct": 36.79503105590062,
  "max_gpu_util_pct": 55.0,
  "avg_gpu_mem_mb": 4134.6459627329195,
  "max_gpu_mem_mb": 4150.0
}{
  "backend": "vllm",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 100,
  "batch_size": 32,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.1180070548599997,
  "avg_batch_latency_s": 2.9501583009999877,
  "throughput_prompts_per_s": 8.474069632416233,
  "throughput_out

In [17]:
# full: base + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/base_hf_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_hf_full


Loaded 1000 prompts
Running Hugging Face benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 435.12it/s]
{
  "backend": "hf",
  "model_path": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "num_prompts": 1000,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.6887423910689999,
  "avg_batch_latency_s": 5.506712247592008,
  "throughput_prompts_per_s": 1.451921666165917,
  "throughput_output_tokens_per_s": 233.51546541279677,
  "peak_cuda_memory_mb": 3705.791015625,
  "avg_gpu_util_pct": 37.62290076335878,
  "max_gpu_util_pct": 73.0,
  "avg_gpu_mem_mb": 4455.21679389313,
  "max_gpu_mem_mb": 4902.0
}
Results saved to outputs/serving/base_hf_full.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --rel

In [18]:
# full: base + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {BASE_MODEL} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/base_vllm_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name base_vllm_full


Loaded 1000 prompts
Running vLLM benchmark for Qwen/Qwen2.5-Coder-1.5B-Instruct
INFO 04-20 20:07:38 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}
INFO 04-20 20:07:40 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 20:07:40 [model.py:1678] Using max model len 32768
INFO 04-20 20:07:40 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 20:07:40 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 20:07:43 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=18653) INFO 04-20 20:07:57 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen2.5-Coder-1.5B-Instruct', s

In [19]:
# full: runtime aware + hf
!python serving/benchmark_serving.py \
  --backend hf \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 8 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_hf_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_hf_full


Loaded 1000 prompts
Running Hugging Face benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 189.04it/s]
{
  "backend": "hf",
  "model_path": "/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged",
  "num_prompts": 1000,
  "batch_size": 8,
  "max_new_tokens": 256,
  "temperature": 0.2,
  "top_p": 0.95,
  "avg_latency_per_prompt_s": 0.6695119966330003,
  "avg_batch_latency_s": 5.352918561072005,
  "throughput_prompts_per_s": 1.4936252151253984,
  "throughput_output_tokens_per_s": 228.62024992795395,
  "peak_cuda_memory_mb": 3705.791015625,
  "avg_gpu_util_pct": 37.285490196078435,
  "max_gpu_util_pct": 92.0,
  "avg_gpu_mem_mb": 4475.345882352941,
  "max_gpu_mem_mb": 4902.0
}
Results saved to outputs/serving/runtime_aware_hf_full.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently log

In [20]:
# full runtime aware + vllm
!python serving/benchmark_serving.py \
  --backend vllm \
  --model_path {RUNTIME_AWARE_MERGED} \
  --input_path {INPUT_PATH} \
  --limit 1000 \
  --batch_size 32 \
  --output_path {SERVING_OUTPUT_DIR}/runtime_aware_vllm_full.json \
  --use_wandb \
  --wandb_project {WANDB_PROJECT} \
  --run_name runtime_aware_vllm_full


Loaded 1000 prompts
Running vLLM benchmark for /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged
INFO 04-20 20:20:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged'}
INFO 04-20 20:20:48 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-20 20:20:48 [model.py:1678] Using max model len 32768
INFO 04-20 20:20:48 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 20:20:48 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 20:20:50 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=25050) INFO 04-20 20:21:04 [core.py:105] Initializing 

In [22]:
import json
import pandas as pd

paths = [
    "outputs/serving/base_hf_full.json",
    "outputs/serving/base_vllm_full.json",
    "outputs/serving/runtime_aware_hf_full.json",
    "outputs/serving/runtime_aware_vllm_full.json",
]

rows = []
for path in paths:
    with open(path, "r") as f:
        rows.append(json.load(f))

df = pd.DataFrame(rows)
df.to_csv("outputs/serving/serving_full_results.csv", index=False)
df


,backend,model_path,num_prompts,batch_size,max_new_tokens,temperature,top_p,avg_latency_per_prompt_s,avg_batch_latency_s,throughput_prompts_per_s,throughput_output_tokens_per_s,peak_cuda_memory_mb,avg_gpu_util_pct,max_gpu_util_pct,avg_gpu_mem_mb,max_gpu_mem_mb
0,hf,Qwen/Qwen2.5-Coder-1.5B-Instruct,1000,8,256,0.2,0.95,0.688742,5.506712,1.451922,233.515465,3705.791016,37.622901,73.0,4455.216794,4902.0
1,vllm,Qwen/Qwen2.5-Coder-1.5B-Instruct,1000,32,256,0.2,0.95,0.031796,0.993604,31.450586,1781.612794,37602.000000,97.213115,100.0,37500.983607,37602.0
2,hf,/content/drive/MyDrive/efficient-codegen/check...,1000,8,256,0.2,0.95,0.669512,5.352919,1.493625,228.620250,3705.791016,37.285490,92.0,4475.345882,4902.0
3,vllm,/content/drive/MyDrive/efficient-codegen/check...,1000,32,256,0.2,0.95,0.031674,0.989795,31.571594,1891.106930,37614.000000,96.770492,100.0,37510.885246,37614.0


In [23]:
from google.colab import files
files.download("outputs/serving/serving_full_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>